In [1]:
from pathlib import Path
import zipfile
import pandas as pd

In [2]:
project_root = Path.cwd().parent

freddie_dir = project_root / "data" / "raw" / "Freddie_Mac"

list(freddie_dir.glob("*.zip"))

[WindowsPath('e:/capstone_project/mortgage-model-reliability/data/raw/Freddie_Mac/file_headers_july_2026.zip'),
 WindowsPath('e:/capstone_project/mortgage-model-reliability/data/raw/Freddie_Mac/release-47-sample-files.zip'),
 WindowsPath('e:/capstone_project/mortgage-model-reliability/data/raw/Freddie_Mac/sample_2015.zip')]

# Step 1: Select the 2015 sample and inspect what's inside

In [3]:
sample_zip = freddie_dir / "sample_2015.zip"

with zipfile.ZipFile(sample_zip, "r") as z:
    file_names = z.namelist()

file_names

['sample_orig_2015.txt', 'sample_perf_2015.txt']

# Step 2 : Identify the origination and performance files

In [4]:
for name in file_names:
    print(name)

sample_orig_2015.txt
sample_perf_2015.txt


# Step 3 : Load the 2015 origination data

In [5]:
with zipfile.ZipFile(sample_zip, "r") as z:
    with z.open("sample_orig_2015.txt") as f:
        orig_2015 = pd.read_csv(
            f,
            sep="|",
            header=None
        )

orig_2015.shape

(50000, 31)

# Step 4 — Quick visual inspection

In [6]:
orig_2015.head()

,0,1,2,3,4,5,6,7,8,9,...,21,22,23,24,25,26,27,28,29,30
0,734,201504,N,203003,NaN,0,1,P,80,15,...,180,2,OTHER,N,NaN,NaN,N,7,N,9999
1,797,201504,N,204503,NaN,0,1,S,37,26,...,360,2,OTHER,N,NaN,NaN,N,7,N,9999
2,670,201504,N,203003,11460.0,12,1,P,90,33,...,180,2,OTHER,N,NaN,NaN,N,7,N,9999
3,731,201503,N,204502,NaN,0,1,I,73,999,...,360,2,OTHER,N,A07Q20031002,NaN,Y,7,N,9999
4,668,201504,N,203003,39340.0,0,2,I,75,30,...,180,2,OTHER,N,NaN,NaN,N,7,N,9999


In [7]:
orig_2015.iloc[0].tolist()

[np.int64(734),
 np.int64(201504),
 'N',
 np.int64(203003),
 np.float64(nan),
 np.int64(0),
 np.int64(1),
 'P',
 np.int64(80),
 np.int64(15),
 np.int64(417000),
 np.int64(65),
 np.float64(2.5),
 'R',
 'N',
 'FRM',
 'CO',
 'SF',
 np.int64(812),
 'F15Q10000025',
 'P',
 np.int64(180),
 np.int64(2),
 'OTHER',
 'N',
 nan,
 nan,
 'N',
 np.int64(7),
 'N',
 np.int64(9999)]

# Step 5 — Define the 31 origination columns - Based on the Freddie Mac origination file layout we reviewed earlier

In [8]:
orig_columns = [
    "credit_score",
    "first_payment_date",
    "first_time_homebuyer_flag",
    "maturity_date",
    "msa",
    "mi_percent",
    "number_of_units",
    "occupancy_status",
    "original_cltv",
    "original_dti",
    "original_upb",
    "original_ltv",
    "original_interest_rate",
    "channel",
    "prepayment_penalty_mortgage_flag",
    "amortization_type",
    "property_state",
    "property_type",
    "postal_code",
    "loan_identifier",
    "loan_purpose",
    "original_loan_term",
    "number_of_borrowers",
    "seller_name",
    "servicer_name",
    "super_conforming_flag",
    "pre_harp_loan_sequence_number",
    "program_indicator",
    "harp_indicator",
    "property_valuation_method",
    "interest_only_indicator"
]

# sanity check:

In [9]:
len(orig_columns)

31

# Assining 

In [10]:
orig_2015.columns = orig_columns

In [11]:
orig_2015.head()

,credit_score,first_payment_date,first_time_homebuyer_flag,maturity_date,msa,mi_percent,number_of_units,occupancy_status,original_cltv,original_dti,...,original_loan_term,number_of_borrowers,seller_name,servicer_name,super_conforming_flag,pre_harp_loan_sequence_number,program_indicator,harp_indicator,property_valuation_method,interest_only_indicator
0,734,201504,N,203003,NaN,0,1,P,80,15,...,180,2,OTHER,N,NaN,NaN,N,7,N,9999
1,797,201504,N,204503,NaN,0,1,S,37,26,...,360,2,OTHER,N,NaN,NaN,N,7,N,9999
2,670,201504,N,203003,11460.0,12,1,P,90,33,...,180,2,OTHER,N,NaN,NaN,N,7,N,9999
3,731,201503,N,204502,NaN,0,1,I,73,999,...,360,2,OTHER,N,A07Q20031002,NaN,Y,7,N,9999
4,668,201504,N,203003,39340.0,0,2,I,75,30,...,180,2,OTHER,N,NaN,NaN,N,7,N,9999


In [12]:
orig_2015.shape

(50000, 31)

# Step 6 — Ingest the 2015 Performance Data

In [13]:
with zipfile.ZipFile(sample_zip, "r") as z:
    with z.open("sample_perf_2015.txt") as f:
        perf_2015 = pd.read_csv(
            f,
            sep="|",
            header=None,
            low_memory=False
        )

perf_2015.shape

(3467166, 35)

# Step 7 — Inspect one raw performance record

In [14]:
perf_2015.head()

,0,1,2,3,4,5,6,7,8,9,...,25,26,27,28,29,30,31,32,33,34
0,F15Q10000025,201503,417000.0,00,0,180,NaN,NaN,NaN,NaN,...,999,NaN,NaN,NaN,NaN,NaN,417000.0,7,U.S. BANK N.A.,NaN
1,F15Q10000025,201504,415000.0,00,1,179,NaN,NaN,NaN,NaN,...,999,NaN,NaN,NaN,NaN,NaN,415000.0,7,U.S. BANK N.A.,NaN
2,F15Q10000025,201505,413000.0,00,2,178,NaN,NaN,NaN,NaN,...,999,NaN,NaN,NaN,NaN,NaN,413000.0,7,U.S. BANK N.A.,NaN
3,F15Q10000025,201506,409000.0,00,3,177,NaN,NaN,NaN,NaN,...,999,NaN,NaN,NaN,NaN,NaN,409000.0,7,U.S. BANK N.A.,NaN
4,F15Q10000025,201507,409000.0,00,4,176,NaN,NaN,NaN,NaN,...,999,NaN,NaN,NaN,NaN,NaN,409000.0,7,U.S. BANK N.A.,NaN


In [15]:
perf_2015.iloc[0].tolist()

['F15Q10000025',
 np.int64(201503),
 np.float64(417000.0),
 '00',
 np.int64(0),
 np.int64(180),
 np.float64(nan),
 nan,
 np.float64(nan),
 np.float64(nan),
 np.float64(2.5),
 np.float64(0.0),
 np.float64(nan),
 np.float64(nan),
 np.float64(nan),
 np.float64(nan),
 np.float64(nan),
 np.float64(nan),
 np.float64(nan),
 np.float64(nan),
 np.float64(nan),
 np.float64(nan),
 np.float64(nan),
 nan,
 nan,
 np.int64(999),
 np.float64(nan),
 np.float64(nan),
 nan,
 nan,
 np.float64(nan),
 np.float64(417000.0),
 '7',
 'U.S. BANK N.A.',
 np.float64(nan)]

# Step 8 — Assign the 35 performance column names

In [16]:
perf_columns = [
    "loan_identifier",
    "period",
    "current_actual_upb",
    "current_loan_delinquency_status",
    "loan_age",
    "remaining_months_to_legal_maturity",
    "defect_settlement_date",
    "modification_flag",
    "zero_balance_code",
    "zero_balance_effective_date",
    "current_interest_rate",
    "current_non_interest_bearing_upb",
    "ddlpi",
    "mi_recoveries",
    "net_sales_proceeds",
    "non_mi_recoveries",
    "total_expenses",
    "legal_costs",
    "maintenance_preservation_costs",
    "taxes_and_insurance",
    "miscellaneous_expenses",
    "actual_loss",
    "cumulative_modification_costs",
    "interest_rate_step_indicator",
    "payment_deferral_flag",
    "estimated_ltv",
    "zero_balance_removal_upb",
    "delinquent_accrued_interest",
    "delinquency_due_to_disaster",
    "borrower_assistance_status_code",
    "current_month_modification_cost",
    "interest_bearing_upb",
    "alternative_delinquency_resolution",
    "servicing_activity_indicator",
    "current_servicer"
]

# sanity check

In [17]:
len(perf_columns)

35

In [18]:
perf_2015.columns = perf_columns

In [19]:
perf_2015.head()

,loan_identifier,period,current_actual_upb,current_loan_delinquency_status,loan_age,remaining_months_to_legal_maturity,defect_settlement_date,modification_flag,zero_balance_code,zero_balance_effective_date,...,estimated_ltv,zero_balance_removal_upb,delinquent_accrued_interest,delinquency_due_to_disaster,borrower_assistance_status_code,current_month_modification_cost,interest_bearing_upb,alternative_delinquency_resolution,servicing_activity_indicator,current_servicer
0,F15Q10000025,201503,417000.0,00,0,180,NaN,NaN,NaN,NaN,...,999,NaN,NaN,NaN,NaN,NaN,417000.0,7,U.S. BANK N.A.,NaN
1,F15Q10000025,201504,415000.0,00,1,179,NaN,NaN,NaN,NaN,...,999,NaN,NaN,NaN,NaN,NaN,415000.0,7,U.S. BANK N.A.,NaN
2,F15Q10000025,201505,413000.0,00,2,178,NaN,NaN,NaN,NaN,...,999,NaN,NaN,NaN,NaN,NaN,413000.0,7,U.S. BANK N.A.,NaN
3,F15Q10000025,201506,409000.0,00,3,177,NaN,NaN,NaN,NaN,...,999,NaN,NaN,NaN,NaN,NaN,409000.0,7,U.S. BANK N.A.,NaN
4,F15Q10000025,201507,409000.0,00,4,176,NaN,NaN,NaN,NaN,...,999,NaN,NaN,NaN,NaN,NaN,409000.0,7,U.S. BANK N.A.,NaN


# cross-file validation

In [20]:
orig_2015["loan_identifier"].nunique()

50000

In [21]:
perf_2015["loan_identifier"].nunique()

50000

# Step 9 — Verify the loan IDs actually match

In [22]:
orig_ids = set(orig_2015["loan_identifier"])
perf_ids = set(perf_2015["loan_identifier"])

len(orig_ids - perf_ids), len(perf_ids - orig_ids)

(0, 0)

# Understand and Validate the 2015 Data

# Step 1: Check the basic structure

In [23]:
print("Origination shape:", orig_2015.shape)
print("Performance shape:", perf_2015.shape)

print("\nOrigination unique loans:",
      orig_2015["loan_identifier"].nunique())

print("Performance unique loans:",
      perf_2015["loan_identifier"].nunique())

Origination shape: (50000, 31)
Performance shape: (3467166, 35)

Origination unique loans: 50000
Performance unique loans: 50000


# step 2: validate the performance timeline  reporting period range

In [24]:
print("Earliest performance period:", perf_2015["period"].min())
print("Latest performance period:", perf_2015["period"].max())

Earliest performance period: 201501
Latest performance period: 202603


# step 3: number of observations per loan

In [25]:
loan_month_counts = perf_2015.groupby("loan_identifier").size()

loan_month_counts.describe()

count    50000.000000
mean        69.343320
std         40.254482
min          1.000000
25%         36.000000
50%         64.000000
75%        112.000000
max        135.000000
dtype: float64

# step 4: Validate the loan-month structure:  for loan_identifier + period -> there should ideally be only one row. 

In [26]:
duplicate_loan_months = perf_2015.duplicated(
    subset=["loan_identifier", "period"]
).sum()

print("Duplicate loan-period rows:", duplicate_loan_months)

Duplicate loan-period rows: 0


In [27]:
loan_month_counts.head(10)

loan_identifier
F15Q10000025     68
F15Q10000031     17
F15Q10000084    133
F15Q10000087     52
F15Q10000157    133
F15Q10000167    134
F15Q10000188    133
F15Q10000230     45
F15Q10000270    134
F15Q10000275     70
dtype: int64

# Check missing values in origination data

In [28]:
orig_missing = (
    orig_2015.isnull()
    .sum()
    .sort_values(ascending=False)
)

orig_missing

pre_harp_loan_sequence_number       49675
super_conforming_flag               45983
msa                                  5604
first_time_homebuyer_flag               0
maturity_date                           0
first_payment_date                      0
credit_score                            0
occupancy_status                        0
original_cltv                           0
original_dti                            0
original_upb                            0
original_ltv                            0
original_interest_rate                  0
mi_percent                              0
number_of_units                         0
prepayment_penalty_mortgage_flag        0
channel                                 0
amortization_type                       0
property_state                          0
loan_identifier                         0
loan_purpose                            0
property_type                           0
postal_code                             0
number_of_borrowers               

In [29]:
orig_2015.dtypes

credit_score                          int64
first_payment_date                    int64
first_time_homebuyer_flag               str
maturity_date                         int64
msa                                 float64
mi_percent                            int64
number_of_units                       int64
occupancy_status                        str
original_cltv                         int64
original_dti                          int64
original_upb                          int64
original_ltv                          int64
original_interest_rate              float64
channel                                 str
prepayment_penalty_mortgage_flag        str
amortization_type                       str
property_state                          str
property_type                           str
postal_code                           int64
loan_identifier                         str
loan_purpose                            str
original_loan_term                    int64
number_of_borrowers             

# inspect the actual origination feature distributions

In [30]:
key_numeric_features = [
    "credit_score",
    "original_dti",
    "original_ltv",
    "original_cltv",
    "original_upb",
    "original_interest_rate",
    "original_loan_term",
    "number_of_borrowers"
]

orig_2015[key_numeric_features].describe().T

,count,mean,std,min,25%,50%,75%,max
credit_score,50000.0,749.758940,47.508044,462.000,718.00,760.0,788.00,834.0
original_dti,50000.0,111.437720,262.595519,1.000,28.00,36.0,43.00,999.0
original_ltv,50000.0,73.528480,17.996670,5.000,65.00,78.0,83.00,999.0
original_cltv,50000.0,74.420920,18.488844,5.000,66.00,79.0,85.00,999.0
original_upb,50000.0,224006.060000,119990.497697,10000.000,131000.00,200000.0,299000.00,974000.0
original_interest_rate,50000.0,3.974625,0.451811,2.375,3.75,4.0,4.25,5.5
original_loan_term,50000.0,315.294020,75.922712,96.000,240.00,360.0,360.00,366.0
number_of_borrowers,50000.0,1.510800,0.499888,1.000,1.00,2.0,2.00,2.0


# Investigation of 999 values 

In [31]:
for col in ["original_dti","original_ltv","original_cltv"]:
    print("\n")
    print(orig_2015[col].value_counts().sort_index().tail(10))



original_dti
42     1868
43     1988
44     2113
45     1901
46      810
47      736
48      823
49      689
50      682
999    4020
Name: count, dtype: int64


original_ltv
175    2
176    1
181    1
182    1
183    1
197    1
200    1
209    1
241    1
999    1
Name: count, dtype: int64


original_cltv
193    1
197    1
200    3
203    1
209    1
219    1
226    1
241    1
854    1
999    1
Name: count, dtype: int64


In [32]:
dti_raw = orig_2015["original_dti"]

dti_valid = orig_2015.loc[
    orig_2015["original_dti"] != 999,
    "original_dti"
]

# comparing stats for dti column

In [33]:
print("INCLUDING 999")
print("Mean:", dti_raw.mean())
print("Median:", dti_raw.median())
print("Std Dev:", dti_raw.std())

print("\nEXCLUDING 999")
print("Mean:", dti_valid.mean())
print("Median:", dti_valid.median())
print("Std Dev:", dti_valid.std())

INCLUDING 999
Mean: 111.43772
Median: 36.0
Std Dev: 262.59551871524883

EXCLUDING 999
Mean: 33.838755980861244
Median: 35.0
Std Dev: 9.391886988255283


# quantifying exactly what proportion of 2015 borrowers have unavailable DTI:

In [34]:
dti_missing_count = (orig_2015["original_dti"] == 999).sum()

dti_missing_pct = (
    (orig_2015["original_dti"] == 999).mean() * 100
)

print("DTI 999 count:", dti_missing_count)
print("DTI 999 percentage:", dti_missing_pct)

DTI 999 count: 4020
DTI 999 percentage: 8.04


as per our statistics and also as per the official documentation, 999 means NOT AVAILABLE so we can clean it - not overwritng the raw data. creating new column after cleaning

In [35]:
import numpy as np

In [36]:
orig_2015["original_dti_clean"]=orig_2015["original_dti"].replace(999,np.nan)


In [37]:
(orig_2015["original_dti"] == 999).sum()

np.int64(4020)

In [38]:
orig_2015["original_dti_clean"].isna().sum()
# previously whereevere we had 999 now we have Nan 

np.int64(4020)

# Investigating original ltv

In [39]:
orig_2015["original_ltv"].describe()

count    50000.00000
mean        73.52848
std         17.99667
min          5.00000
25%         65.00000
50%         78.00000
75%         83.00000
max        999.00000
Name: original_ltv, dtype: float64

# Inspecting what values occur above 100. LTI above 100 is unsual 

In [41]:
orig_2015.loc[
    orig_2015["original_ltv"] > 100,
    "original_ltv"
].value_counts().sort_index()

original_ltv
101    12
102    30
103    38
104    18
105    25
       ..
197     1
200     1
209     1
241     1
999     1
Name: count, Length: 79, dtype: int64

# isolating the documented sentinel 999

In [ ]:
ltv_raw = orig_2015["original_ltv"]
ltv_clean = orig_2015.loc[
    orig_2015["original_ltv"] !=999,
    "original_ltv"
]

print("INCLUDING 999")
print
)